# ML Cross-Check — IsolationForest

Everything in `02_anomaly_detection.ipynb` is rule-based (a hypothesis turned into a boolean filter). This notebook runs an unsupervised model with no rules at all, to check whether the 4 confirmed clusters are the whole picture — or just the most obvious part of it.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                      'axes.spines.right': False, 'figure.facecolor': 'white'})

DATA_PATH = '/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv'

df = pd.read_csv(DATA_PATH)
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])
from sklearn.ensemble import IsolationForest

df['latency_sec'] = (df['processed_at'] - df['created_at']).dt.total_seconds()
df['is_fail'] = (df['status'] == 'fail').astype(int)
df['amount_zscore'] = df.groupby('currency')['amount'].transform(lambda x: (x - x.mean()) / x.std())
refund_gap = df['refunded_amount'] - df['amount']
df['refund_gap_zscore'] = refund_gap.groupby(df['currency']).transform(lambda x: (x - x.mean()) / x.std())

known = {
    'bank_id==777': df['bank_id'] == 777,
    'psp_gamma window': (df['psp_id'] == 'psp_gamma') & df['created_at'].between('2025-05-12', '2025-05-17 23:59:59'),
    'psp_beta over-refund': df['refunded_amount'] > df['amount'],
    'fail_refund': df['has_refund'] & (df['status'] == 'fail'),
}

In [2]:
X = df[['latency_sec', 'amount_zscore', 'is_fail', 'is_secured', 'refund_gap_zscore']]
model = IsolationForest(random_state=42, contamination='auto').fit(X)
df['if_score'] = model.decision_function(X)   # lower = more anomalous

print(f"{'cluster':<24}{'mean score':>12}{'vs global':>12}")
global_mean = df['if_score'].mean()
print(f"{'(global)':<24}{global_mean:>12.4f}{'—':>12}")
for name, mask in known.items():
    m = df.loc[mask, 'if_score'].mean()
    flag = 'DETECTED' if m < global_mean - 0.02 else 'missed'
    print(f"{name:<24}{m:>12.4f}{flag:>12}")

cluster                   mean score   vs global
(global)                      0.0283           —
bank_id==777                  0.0346      missed
psp_gamma window             -0.1150    DETECTED
psp_beta over-refund         -0.0268    DETECTED
fail_refund                  -0.0261    DETECTED


- **psp_gamma, psp_beta, fail_refund** all score well below the global mean — the model independently rediscovers 3 of the 4 clusters using only generic engineered features (latency, currency-normalized amount/refund, fail flag), no rules
- **`bank_id == 777` is missed** — its signature is zero variance *within* a group of 635 rows, a **collective anomaly**. A model that scores each row independently has nothing to detect: a lone 5-second transaction looks completely ordinary

In [3]:
threshold = df['if_score'].quantile(0.01)
flagged = df['if_score'] <= threshold
is_known = pd.concat(known.values(), axis=1).any(axis=1)
new = flagged & ~is_known

print(f"Top 1% most anomalous by IsolationForest: {flagged.sum():,} rows")
print(f"  overlap with the 4 known clusters: {(flagged & is_known).sum():,}")
print(f"  outside all known clusters:        {new.sum():,}")
print(df.loc[new, 'psp_id'].value_counts().rename('rows').to_frame().T)

Top 1% most anomalous by IsolationForest: 10,006 rows
  overlap with the 4 known clusters: 424
  outside all known clusters:        9,582
psp_id  psp_gamma  psp_beta  psp_alpha
rows         3372      3354       2856


- the residual ~9.5k rows outside known clusters are balanced across `psp_id`/`bank_id`/`currency` — a diffuse mix of borderline cases, not a shared fingerprint like the 4 confirmed clusters
- **conclusion:** treat the model as a validation check, not a source of extra labels — it confirms 3/4 rule-based clusters and correctly cannot see the 4th (a structural blind spot of row-independent scoring, not a feature-engineering gap)